# Experiment Tracker Demo

This notebook demonstrates how to use the `ExperimentTracker` module in caketool for tracking ML experiments.

## Features

- **Unified API**: Same interface for different backends (Vertex AI, MLflow)
- **Context Manager**: Clean resource management with `with` statement
- **Mode Support**: `develop` mode for training, `deploy` mode for production (no logging)
- **Artifact Logging**: Log parameters, metrics, files, and pickled objects

In [ ]:
# Install dependencies (uncomment if needed)
!pip install mlflow xgboost scikit-learn

## Start MLflow UI

Before running the experiments, start the MLflow tracking server:

```bash
mlflow ui --host 127.0.0.1 --port 5000
```

Then open http://127.0.0.1:5000 in your browser to view experiments.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from caketool.experiment import create_tracker, MLflowTracker

## 1. Basic Usage with MLflow

The simplest way to track experiments is using the factory function `create_tracker`.

In [ ]:
# Create sample data
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_classes=2,
    random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create tracker using factory function
tracker = create_tracker(
    backend="mlflow",
    experiment_name="random-forest-demo",
    run_name="run-001",
    tracking_uri="http://127.0.0.1:5000/",
    tags={"model": "random_forest", "dataset": "synthetic"},
)

# Use context manager for clean resource management
with tracker:
    # Define hyperparameters
    params = {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 2,
        "random_state": 42,
    }
    
    # Log parameters
    tracker.log_params(params)
    
    # Train model
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Log metrics
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_proba),
    }
    tracker.log_metrics(metrics)
    
    # Log model as pickle
    tracker.log_pickle(model, "random_forest_model")
    
    print("Logged metrics:")
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")

## 2. Time-Series Metrics (Training Progress)

Use the `step` parameter to log metrics over training iterations.

In [ ]:
from sklearn.linear_model import SGDClassifier

tracker = create_tracker(
    backend="mlflow",
    experiment_name="sgd-training-progress",
    run_name="incremental-training",
    tracking_uri="http://127.0.0.1:5000/",
)

with tracker:
    params = {
        "loss": "log_loss",
        "learning_rate": "adaptive",
        "eta0": 0.1,
    }
    tracker.log_params(params)
    
    # Initialize model
    model = SGDClassifier(**params, random_state=42, warm_start=True)
    
    # Incremental training with progress logging
    n_epochs = 10
    for epoch in range(n_epochs):
        model.partial_fit(X_train, y_train, classes=[0, 1])
        
        # Evaluate at each epoch
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        
        # Log metrics with step
        tracker.log_metrics(
            {
                "train_accuracy": train_score,
                "test_accuracy": test_score,
            },
            step=epoch,
        )
        
        print(f"Epoch {epoch + 1}/{n_epochs}: train={train_score:.4f}, test={test_score:.4f}")

## 3. Direct Class Usage

You can also instantiate tracker classes directly.

In [ ]:
# Direct instantiation of MLflowTracker
tracker = MLflowTracker(
    experiment_name="direct-usage-demo",
    run_name="manual-run",
    tracking_uri="http://127.0.0.1:5000/",
    artifact_location=None,  # Default location
    tags={"version": "1.0"},
)

with tracker:
    tracker.log_params({"model_type": "demo"})
    tracker.log_metrics({"demo_metric": 0.99})
    print("Direct usage completed!")

## 4. Deploy Mode (Production)

In production, use `mode='deploy'` to disable all logging operations.

In [ ]:
# Deploy mode - no logging happens
tracker = create_tracker(
    backend="mlflow",
    experiment_name="production-model",
    run_name="inference",
    tracking_uri="http://127.0.0.1:5000/",
    mode="deploy",  # Disables all logging
)

with tracker:
    # These calls do nothing in deploy mode
    tracker.log_params({"ignored": True})
    tracker.log_metrics({"also_ignored": 1.0})
    
    # Your inference code here
    print("Running in deploy mode - no logging overhead!")

## 5. Vertex AI Tracker (Google Cloud)

For Google Cloud environments, use the Vertex AI backend.

```python
from caketool.experiment import create_tracker

tracker = create_tracker(
    backend="vertex_ai",
    experiment_name="my-gcp-experiment",
    run_name="training-run-001",
    project="your-gcp-project-id",
    location="us-central1",
    bucket_name="your-gcs-bucket",
    experiment_description="XGBoost model training",
    experiment_tensorboard=False,
)

with tracker:
    tracker.log_params({"learning_rate": 0.1})
    tracker.log_metrics({"gini": 0.45})
    tracker.log_pickle(model, "trained_model")
```

## 6. Complete ML Pipeline Example

Putting it all together in a realistic scenario.

In [ ]:
from datetime import datetime

def train_model_with_tracking(
    X_train,
    y_train,
    X_test,
    y_test,
    params: dict,
    experiment_name: str = "ml-pipeline",
    mode: str = "develop",
):
    """
    Train a model with full experiment tracking.
    
    Parameters
    ----------
    X_train, y_train : array-like
        Training data.
    X_test, y_test : array-like
        Test data.
    params : dict
        Model hyperparameters.
    experiment_name : str
        Name of the experiment.
    mode : str
        "develop" for training, "deploy" for production.
    
    Returns
    -------
    model
        Trained model.
    dict
        Evaluation metrics.
    """
    run_name = f"run-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    
    tracker = create_tracker(
        backend="mlflow",
        experiment_name=experiment_name,
        run_name=run_name,
        tracking_uri="http://127.0.0.1:5000/",
        mode=mode,
        tags={"pipeline": "automated"},
    )
    
    with tracker:
        # Log parameters
        tracker.log_params(params)
        tracker.log_params({
            "train_size": len(X_train),
            "test_size": len(X_test),
            "n_features": X_train.shape[1],
        })
        
        # Train
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "auc": roc_auc_score(y_test, y_proba),
        }
        tracker.log_metrics(metrics)
        
        # Log model
        tracker.log_pickle(model, "model")
        
    return model, metrics


# Run the pipeline
params = {
    "n_estimators": 50,
    "max_depth": 8,
    "random_state": 42,
}

model, metrics = train_model_with_tracking(
    X_train, y_train, X_test, y_test,
    params=params,
    experiment_name="pipeline-demo",
)

print("\nPipeline completed!")
print(f"Metrics: {metrics}")